# Ontology Agent Evaluation — Scores Overview

Analysis of AI coding agent performance across 4 biomedical ontologies,
comparing Claude Code and OpenAI Codex runtimes with multiple model tiers.

In [ ]:
import os
# Ensure we're at repo root regardless of where jupyter runs
while not os.path.exists('analysis/scores.tsv'):
    os.chdir('..')
    if os.getcwd() == '/':
        raise RuntimeError('Could not find analysis/scores.tsv')
print(f'Working directory: {os.getcwd()}')

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ai4c_scribe.analysis import (
    load_scores, load_reviews, summary_by_model, summary_by_runtime,
    pivot_scores, rubric_summary, outcome_distribution, failure_mode_counts,
)

sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
pd.set_option('display.precision', 3)

df = load_scores(Path('analysis/scores.tsv'))
reviews = load_reviews()
print(f'{len(df)} scored runs, {len(reviews)} qualitative reviews')
print(f'Ontologies: {sorted(df["ontology"].unique())}')
print(f'Models: {sorted(df["model"].unique())}')

## 1. Model Comparison

In [ ]:
summary_by_model(df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
order = df.groupby('model')['f1'].mean().sort_values(ascending=False).index
sns.boxplot(data=df, x='model', y='f1', order=order, ax=ax)
sns.stripplot(data=df, x='model', y='f1', order=order, color='black', alpha=0.5, size=6, ax=ax)
ax.set_title('F1 Score Distribution by Model')
ax.set_xlabel('')
ax.set_ylabel('Metadiff F1')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('analysis/notebooks/fig_f1_by_model.png', dpi=150)
plt.show()

## 2. Runtime Comparison (Claude Code vs Codex)

In [ ]:
summary_by_runtime(df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Box plot by runtime
sns.boxplot(data=df, x='runtime', y='f1', ax=axes[0])
sns.stripplot(data=df, x='runtime', y='f1', color='black', alpha=0.5, ax=axes[0])
axes[0].set_title('F1 by Runtime')
axes[0].set_ylabel('Metadiff F1')

# By runtime and difficulty
sns.barplot(data=df, x='difficulty', y='f1', hue='runtime',
            order=['simple', 'medium', 'hard'], ax=axes[1])
axes[1].set_title('F1 by Difficulty and Runtime')
axes[1].set_ylabel('Mean F1')
axes[1].legend(title='Runtime')

plt.tight_layout()
plt.savefig('analysis/notebooks/fig_f1_by_runtime.png', dpi=150)
plt.show()

## 3. Ontology × Model Heatmap

In [ ]:
pivot = pivot_scores(df, rows='ontology', cols='model')
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Mean F1'})
ax.set_title('Mean F1 Score: Ontology × Model')
ax.set_ylabel('')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('analysis/notebooks/fig_heatmap_ont_model.png', dpi=150)
plt.show()

## 4. Task Type Analysis

In [ ]:
pivot_scores(df, rows='case_type', cols='runtime')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
task_order = df.groupby('case_type')['f1'].mean().sort_values(ascending=False).index
sns.barplot(data=df, x='case_type', y='f1', hue='runtime', order=task_order, ax=ax)
ax.set_title('Mean F1 by Task Type and Runtime')
ax.set_xlabel('Task Type')
ax.set_ylabel('Mean F1')
ax.legend(title='Runtime')
plt.tight_layout()
plt.savefig('analysis/notebooks/fig_f1_by_task_type.png', dpi=150)
plt.show()

## 5. Skills Ablation Study

In [ ]:
ablation = df[df['agent_config_tag'].isin(['v8', 'v8-noskills', 'v9', 'v2', 'v2-noskills', 'v3'])].copy()
ablation['has_skills'] = ~ablation['agent_config_tag'].str.contains('noskills')

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=ablation, x='runtime', y='f1', hue='has_skills', ax=ax)
ax.set_title('Skills Ablation: F1 With vs Without Skills')
ax.set_ylabel('Mean F1')
ax.legend(title='Skills Available', labels=['No Skills', 'With Skills'])
plt.tight_layout()
plt.savefig('analysis/notebooks/fig_skills_ablation.png', dpi=150)
plt.show()

print('Detail:')
pivot_scores(ablation, rows='agent_config_tag', cols='runtime')

## 6. All Scored Runs

In [ ]:
cols = ['ontology', 'issue_number', 'case_type', 'difficulty',
        'agent_config_tag', 'model', 'runtime', 'f1', 'precision', 'recall']
df[cols].sort_values(['ontology', 'issue_number', 'model']).style.background_gradient(
    subset=['f1', 'precision', 'recall'], cmap='RdYlGn', vmin=0, vmax=1
)

## 7. Qualitative Reviews

In [ ]:
if len(reviews) > 0:
    print(f'{len(reviews)} reviews loaded')
    display(rubric_summary(reviews))
else:
    print('No reviews yet')

In [ ]:
if len(reviews) > 0:
    rubric_cols = ['instruction_following', 'correctness', 'completeness',
                   'scope_discipline', 'methodology', 'overall']
    cols_present = [c for c in rubric_cols if c in reviews.columns]
    if cols_present:
        fig, ax = plt.subplots(figsize=(10, 5))
        melted = reviews.melt(id_vars=['model'], value_vars=cols_present,
                             var_name='rubric', value_name='score')
        sns.barplot(data=melted, x='rubric', y='score', hue='model', ax=ax)
        ax.set_title('Rubric Scores by Model')
        ax.set_ylabel('Score (1-5)')
        ax.set_xlabel('')
        ax.set_ylim(0, 5.5)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
        plt.tight_layout()
        plt.savefig('analysis/notebooks/fig_rubric_scores.png', dpi=150)
        plt.show()

In [ ]:
if len(reviews) > 0:
    print('Outcome distribution:')
    display(outcome_distribution(reviews))
    print('\nFailure modes:')
    display(failure_mode_counts(reviews))

## 8. Precision vs Recall

In [ ]:
nonzero = df[df['f1'] > 0].copy()

fig, ax = plt.subplots(figsize=(8, 8))
sns.scatterplot(data=nonzero, x='recall', y='precision', hue='runtime',
                style='difficulty', s=120, alpha=0.8, ax=ax)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.2)
ax.set_title('Precision vs Recall (non-zero runs)')
ax.set_xlabel('Recall (fraction of human changes reproduced)')
ax.set_ylabel('Precision (fraction of agent changes that match human)')

# Add F1 contours
import numpy as np
for f1_val in [0.2, 0.4, 0.6, 0.8]:
    r = np.linspace(0.01, 1, 100)
    p = (f1_val * r) / (2 * r - f1_val)
    mask = (p > 0) & (p <= 1)
    ax.plot(r[mask], p[mask], 'gray', alpha=0.15)
    ax.annotate(f'F1={f1_val}', xy=(1.0, (f1_val * 1.0) / (2 * 1.0 - f1_val)),
               fontsize=8, color='gray', alpha=0.5)

plt.tight_layout()
plt.savefig('analysis/notebooks/fig_precision_recall.png', dpi=150)
plt.show()